# Customer Segmentation Using Machine Learning-Based Clustering
### End-to-End B.Tech CSE Academic Project

## 1. Project Overview & Objective
Customer segmentation is the process of partitioning a company's customer base into distinct sub-groups based on shared characteristics, behaviors, and financial profiles. In retail and e-commerce, customer segmentation enables personalized marketing, improved customer retention, and optimal resource allocation.

In this project, we implement an **unsupervised machine learning** pipeline centered around the **K-Means Clustering** algorithm to automatically categorize customers based on:
- Demographics (`Age`, `Gender`)
- Financial capability (`Annual_Income`)
- Behavioral indicators (`Spending_Score`, `Purchase_Frequency`, `Average_Order_Value`, `Recency`, `Website_Visits`)
- Channel preference (`Online_Purchases`, `Offline_Purchases`)

## 2. Importing Required Libraries

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Plotting styles
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print("Libraries successfully imported!")

## 3. Ingesting the Customer Dataset

In [ ]:
dataset_path = 'dataset/customers.csv'
df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 4. Data Preprocessing & Cleaning
1. Inspect dataset structure and data types
2. Check for missing values
3. Check for duplicate records
4. Encode categorical variables (`Gender` -> Binary: Female=0, Male=1)

In [ ]:
# Dataset information
df.info()

# Missing values check
print("\nMissing values count:")
print(df.isnull().sum())

# Check duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Numerical descriptive statistics
df.describe().T

In [ ]:
# Encode Gender to binary
df['Gender_Encoded'] = df['Gender'].map({'Female': 0, 'Male': 1})

# Numerical features selected for clustering
features = ['Gender_Encoded', 'Age', 'Annual_Income', 'Spending_Score', 'Purchase_Frequency',
            'Average_Order_Value', 'Total_Purchases', 'Recency',
            'Online_Purchases', 'Offline_Purchases', 'Website_Visits']
X = df[features]
print(f"Selected feature matrix shape: {X.shape}")

## 5. Exploratory Data Analysis (EDA)
Visualizing distribution of income, spending scores, and relationships among customer metrics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['Annual_Income'], kde=True, ax=axes[0, 0], color='#4f46e5')
axes[0, 0].set_title('Annual Income Distribution ($)', fontweight='bold')

sns.histplot(df['Spending_Score'], kde=True, ax=axes[0, 1], color='#06b6d4')
axes[0, 1].set_title('Spending Score Distribution (1-100)', fontweight='bold')

sns.histplot(df['Age'], kde=True, ax=axes[1, 0], color='#10b981')
axes[1, 0].set_title('Age Distribution (Years)', fontweight='bold')

sns.scatterplot(x='Annual_Income', y='Spending_Score', hue='Gender', data=df, ax=axes[1, 1], alpha=0.7)
axes[1, 1].set_title('Annual Income vs Spending Score', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
num_cols = ['Age', 'Annual_Income', 'Spending_Score', 'Purchase_Frequency', 'Average_Order_Value', 'Total_Purchases', 'Recency', 'Website_Visits']
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Matrix of Customer Features', fontweight='bold', fontsize=13)
plt.show()

## 6. Feature Scaling with StandardScaler
K-Means computes Euclidean distances between vectors. Features with larger numeric ranges (e.g., Annual Income: \$15,000 to \$150,000) would dominate features with smaller scales (e.g., Age: 18 to 70 or Spending Score: 1 to 100). Normalizing with `StandardScaler` ($\mu=0, \sigma=1$) is crucial.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Scaled Feature Matrix Mean: {X_scaled.mean():.4f}, Std: {X_scaled.std():.4f}")

## 7. Determining Optimal K: Elbow Method & Silhouette Analysis
We evaluate cluster candidate sizes from $K=2$ to $K=10$ using:
1. **WCSS / Inertia** (Within-Cluster Sum of Squares) for the Elbow Method
2. **Silhouette Score** (Measure of cohesion vs separation)
3. **Davies-Bouldin Index** (Ratio of within-cluster distance to between-cluster separation)
4. **Calinski-Harabasz Score** (Variance Ratio Criterion)

In [ ]:
k_range = range(2, 11)
wcss = []
sil_scores = []
db_scores = []
ch_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=15, max_iter=300, random_state=42)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))

eval_df = pd.DataFrame({
    'K': list(k_range),
    'WCSS (Inertia)': np.round(wcss, 2),
    'Silhouette Score': np.round(sil_scores, 4),
    'Davies-Bouldin Index': np.round(db_scores, 4),
    'Calinski-Harabasz Score': np.round(ch_scores, 2)
})
eval_df

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Elbow Curve
ax1.plot(list(k_range), wcss, marker='o', color='#4f46e5', linewidth=2.5)
ax1.axvline(x=5, color='red', linestyle='--', label='Optimal K=5')
ax1.set_title('Elbow Method (WCSS vs K)', fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.legend()

# Silhouette Curve
ax2.plot(list(k_range), sil_scores, marker='s', color='#06b6d4', linewidth=2.5)
ax2.axvline(x=5, color='red', linestyle='--', label='Optimal K=5')
ax2.set_title('Silhouette Score vs K', fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.legend()

plt.tight_layout()
plt.show()

## 8. Training Final K-Means Model ($K=5$)

In [ ]:
final_k = 5
kmeans = KMeans(n_clusters=final_k, init='k-means++', n_init=25, max_iter=500, random_state=42)
df['Cluster_ID'] = kmeans.fit_predict(X_scaled)
print("Final model trained successfully!")

## 9. Dynamic Cluster Profiling & Segment Interpretation

In [ ]:
summary = df.groupby('Cluster_ID')[['Age', 'Annual_Income', 'Spending_Score', 'Purchase_Frequency', 'Average_Order_Value', 'Recency', 'Online_Purchases', 'Offline_Purchases']].mean().round(2)
summary['Count'] = df['Cluster_ID'].value_counts().sort_index()
summary['Percentage'] = (summary['Count'] / len(df) * 100).round(2)

# Dynamic labeling rule (INR ₹)
def label_segment(row):
    if row['Annual_Income'] >= 1500000 and row['Spending_Score'] >= 60:
        return "High-Value Champions"
    elif row['Annual_Income'] >= 1500000 and row['Spending_Score'] < 45:
        return "Affluent Conservative Savers"
    elif row['Annual_Income'] < 1200000 and row['Spending_Score'] >= 65 and row['Age'] < 35:
        return "Young Impulse Trendsetters"
    elif row['Recency'] >= 100:
        return "At-Risk Occasional Customers"
    elif row['Annual_Income'] < 700000 and row['Spending_Score'] < 50:
        return "Budget-Conscious Shoppers"
    else:
        return "Steady Core Customers"

summary['Segment_Name'] = summary.apply(label_segment, axis=1)
df['Segment_Name'] = df['Cluster_ID'].map(summary['Segment_Name'])
summary[['Segment_Name', 'Count', 'Percentage', 'Annual_Income', 'Spending_Score', 'Average_Order_Value', 'Recency']]

## 10. Visualizing Discovered Segments

In [ ]:
# 2D PCA Cluster Projection
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df['PCA1'] = pca_coords[:, 0]
df['PCA2'] = pca_coords[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='Segment_Name', data=df, palette='tab10', s=70, alpha=0.8, edgecolor='black', linewidth=0.5)
plt.title(f'2D PCA Projection of Customer Clusters (Variance: {pca.explained_variance_ratio_.sum()*100:.1f}%)', fontweight='bold')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 11. Model Persistence (Saving with Joblib)

In [ ]:
os.makedirs('model', exist_ok=True)
joblib.dump(kmeans, 'model/kmeans_model.pkl')
joblib.dump(scaler, 'model/scaler.pkl')
print("Trained model and scaler successfully serialized!")

## 12. Inference on New Customer
Example prediction simulating the Flask web app backend.

In [ ]:
# Example new customer input
new_customer = pd.DataFrame([{
    'Gender_Encoded': 0, # Female
    'Age': 24,
    'Annual_Income': 720000,
    'Spending_Score': 85,
    'Purchase_Frequency': 22,
    'Average_Order_Value': 2800,
    'Total_Purchases': 55,
    'Recency': 10,
    'Online_Purchases': 45,
    'Offline_Purchases': 10,
    'Website_Visits': 38
}])

# Scale using fitted scaler
scaled_input = scaler.transform(new_customer)
predicted_cluster = int(kmeans.predict(scaled_input)[0])
predicted_name = summary.loc[predicted_cluster, 'Segment_Name']

print(f"-> Customer belongs to Cluster {predicted_cluster}: {predicted_name}")